# Hazır Modellerle Futbol Video Analizi Test Notebooku

Bu notebookta **yeni model eğitimi yoktur**. Amaç hazır modellerle şu kontrolleri yapmaktır:

1. Model oyuncuları görsellerde / frame'lerde doğru yakalıyor mu?
2. Model videoda oyuncuları kaçırmadan yakalıyor mu?
3. Tracking ile oyuncu ID'leri birkaç saniye korunabiliyor mu?

Sonraki aşamalar için altyapı da eklenmiştir:

- Takım ayrımı
- Saha koordinatına dönüştürme
- Heatmap
- Mesafe / hız analizi

> Not: Hazır COCO modellerinde sınıf adı `person`dır. Bu projede `person` çıktısını futbolcu/oyuncu olarak kullanıyoruz. Bu yüzden `classes=[0]` filtresi kullanılır.

## 1. Kurulum

Kaggle Notebook üzerinde çalıştırılmak üzere hazırlanmıştır. İnternet kapalıysa Kaggle settings üzerinden Internet seçeneğini açman gerekebilir.

In [ ]:
!pip install -q ultralytics opencv-python-headless scikit-learn pandas matplotlib pyyaml

## 2. Importlar ve temel ayarlar

In [ ]:
import os
import cv2
import time
import math
import yaml
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from sklearn.cluster import KMeans
from ultralytics import YOLO

pd.set_option('display.max_columns', 100)

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/ready_model_video_analysis')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print('Kaggle input klasörleri:')
for p in INPUT_ROOT.iterdir():
    print('-', p)
print('
Çalışma klasörü:', WORK_ROOT)

## 3. Video ve model ayarları

Aşağıdaki `VIDEO_PATH` değerini kendi Kaggle input video yoluna göre değiştir. Önce Kaggle'a kısa bir maç videosu dataset olarak yüklemeni öneririm. İlk test için 10-30 saniyelik video yeterli.

In [ ]:
# KENDİ VİDEO PATH'İNİ BURAYA YAZ
VIDEO_PATH = Path('/kaggle/input/test-football-video/video.mp4')

# Görsel klasörü opsiyonel. Varsa frame/görsel testi için kullanılır.
IMAGE_TEST_DIR = Path('/kaggle/input/test-football-images')

# Hazır modeller. İlk olarak YOLO11n ve YOLOv8n önerilir.
# YOLOv10n bazı ortamlarda destek/indirme sorunları çıkarırsa listeden kaldırabilirsin.
MODEL_NAMES = [
    'yolo11n.pt',
    'yolov8n.pt',
    'yolov10n.pt'
]

# COCO person class id = 0
PERSON_CLASS_ID = 0

# Test ayarları
CONF_VALUES = [0.15, 0.25, 0.35]
IMG_SIZE_VALUES = [640, 960]
TRACKERS = ['bytetrack.yaml', 'botsort.yaml']

# Uzun video için işlem sınırı. İlk testte düşük tutmak mantıklı.
MAX_BENCHMARK_FRAMES = 300
MAX_TRACK_ANALYSIS_FRAMES = 500

print('VIDEO_PATH exists:', VIDEO_PATH.exists(), VIDEO_PATH)
print('IMAGE_TEST_DIR exists:', IMAGE_TEST_DIR.exists(), IMAGE_TEST_DIR)

## 4. Yardımcı fonksiyonlar

In [ ]:
def load_available_models(model_names):
    models = {}
    failed = {}
    for name in model_names:
        try:
            print(f'Yükleniyor: {name}')
            models[name] = YOLO(name)
            print(f'  OK: {name}')
        except Exception as e:
            failed[name] = str(e)
            print(f'  HATA: {name} -> {e}')
    return models, failed


def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f'Video açılamadı: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps else 0
    cap.release()
    return {
        'fps': fps,
        'frame_count': frame_count,
        'width': width,
        'height': height,
        'duration_sec': duration
    }


def extract_sample_frames(video_path, out_dir, every_n_frames=60, max_frames=30):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    saved = []
    idx = 0
    while cap.isOpened() and len(saved) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % every_n_frames == 0:
            path = out_dir / f'frame_{idx:06d}.jpg'
            cv2.imwrite(str(path), frame)
            saved.append(path)
        idx += 1
    cap.release()
    print(f'{len(saved)} frame kaydedildi -> {out_dir}')
    return saved


def show_image_bgr(path, size=(12, 7)):
    img = cv2.imread(str(path))
    if img is None:
        print('Görsel okunamadı:', path)
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size)
    plt.imshow(img)
    plt.axis('off')
    plt.show()


def xyxy_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0


def bottom_center_xyxy(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, y2)

## 5. Video bilgisi ve örnek frame çıkarma

In [ ]:
if VIDEO_PATH.exists():
    video_info = get_video_info(VIDEO_PATH)
    display(pd.DataFrame([video_info]))
    sample_dir = WORK_ROOT / 'sample_frames'
    sample_frames = extract_sample_frames(VIDEO_PATH, sample_dir, every_n_frames=60, max_frames=20)
    if sample_frames:
        show_image_bgr(sample_frames[0])
else:
    print('VIDEO_PATH bulunamadı. Önce VIDEO_PATH değişkenini kendi video yoluna göre düzelt.')

## 6. Hazır modelleri yükle

In [ ]:
models, failed_models = load_available_models(MODEL_NAMES)
print('
Yüklenen modeller:', list(models.keys()))
print('Yüklenemeyenler:', failed_models)

## 7. Görsel / frame üzerinde detection testi

Bu hücre örnek frame'lerde hazır modellerin oyuncuları yakalayıp yakalamadığını hızlıca görmeni sağlar.

In [ ]:
DETECTION_IMAGE_OUT = WORK_ROOT / 'image_detection_tests'
DETECTION_IMAGE_OUT.mkdir(parents=True, exist_ok=True)

source_for_image_test = None
if IMAGE_TEST_DIR.exists():
    source_for_image_test = IMAGE_TEST_DIR
elif VIDEO_PATH.exists():
    source_for_image_test = WORK_ROOT / 'sample_frames'

if source_for_image_test is None or not Path(source_for_image_test).exists():
    print('Görsel/frame test kaynağı bulunamadı.')
else:
    for model_name, model in models.items():
        for conf in [0.15, 0.25]:
            run_name = f'{Path(model_name).stem}_conf{str(conf).replace(".","")}'
            print('Predict:', model_name, 'conf:', conf)
            model.predict(
                source=str(source_for_image_test),
                conf=conf,
                imgsz=640,
                classes=[PERSON_CLASS_ID],
                save=True,
                project=str(DETECTION_IMAGE_OUT),
                name=run_name,
                exist_ok=True
            )

## 8. Video üzerinde detection çıktısı üret

Bu hücre farklı `conf` değerleriyle video çıktısı üretir. Önce `imgsz=640` ile başla. Uzak oyuncular çok kaçıyorsa sonra `imgsz=960` dene.

In [ ]:
VIDEO_DETECTION_OUT = WORK_ROOT / 'video_detection_outputs'
VIDEO_DETECTION_OUT.mkdir(parents=True, exist_ok=True)

if not VIDEO_PATH.exists():
    print('Video bulunamadı. VIDEO_PATH değerini düzelt.')
else:
    for model_name, model in models.items():
        for conf in CONF_VALUES:
            run_name = f'{Path(model_name).stem}_img640_conf{str(conf).replace(".","")}'
            print('Video predict:', run_name)
            model.predict(
                source=str(VIDEO_PATH),
                conf=conf,
                imgsz=640,
                classes=[PERSON_CLASS_ID],
                save=True,
                project=str(VIDEO_DETECTION_OUT),
                name=run_name,
                exist_ok=True
            )

## 9. Detection benchmark: FPS ve kişi sayısı istikrarı

Etiketli video olmadığı için gerçek mAP ölçemeyiz. Bunun yerine pratik test yapıyoruz:

- Ortalama inference FPS
- Frame başına ortalama kişi sayısı
- Kişi sayısı standart sapması
- Boş detection frame oranı

Bu değerler kesin başarı metriği değil, ama hazır modelleri karşılaştırmak için kullanışlıdır.

In [ ]:
def benchmark_detection_on_video(model, video_path, model_name, conf=0.25, imgsz=640, max_frames=300):
    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0
    times = []
    counts = []
    confidences = []

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.time()
        results = model.predict(
            source=frame,
            conf=conf,
            imgsz=imgsz,
            classes=[PERSON_CLASS_ID],
            verbose=False
        )
        dt = time.time() - t0
        times.append(dt)
        r = results[0]
        n = 0 if r.boxes is None else len(r.boxes)
        counts.append(n)
        if r.boxes is not None and len(r.boxes) > 0:
            confidences.extend(r.boxes.conf.cpu().numpy().tolist())
        frame_idx += 1

    cap.release()
    total_time = sum(times)
    processed = len(times)
    fps = processed / total_time if total_time > 0 else 0
    return {
        'model': model_name,
        'conf': conf,
        'imgsz': imgsz,
        'processed_frames': processed,
        'benchmark_fps': fps,
        'avg_person_count': float(np.mean(counts)) if counts else 0,
        'std_person_count': float(np.std(counts)) if counts else 0,
        'empty_frame_ratio': float(np.mean([c == 0 for c in counts])) if counts else 0,
        'avg_confidence': float(np.mean(confidences)) if confidences else 0
    }

benchmark_rows = []
if VIDEO_PATH.exists():
    for model_name, model in models.items():
        for conf in CONF_VALUES:
            row = benchmark_detection_on_video(
                model, VIDEO_PATH, model_name,
                conf=conf,
                imgsz=640,
                max_frames=MAX_BENCHMARK_FRAMES
            )
            benchmark_rows.append(row)
            print(row)

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_csv = WORK_ROOT / 'detection_benchmark_results.csv'
benchmark_df.to_csv(benchmark_csv, index=False)
display(benchmark_df)
print('Kaydedildi:', benchmark_csv)

## 10. Benchmark grafiklerini çiz

In [ ]:
if 'benchmark_df' in globals() and len(benchmark_df) > 0:
    plt.figure(figsize=(10, 5))
    labels = benchmark_df['model'] + ' conf=' + benchmark_df['conf'].astype(str)
    plt.bar(labels, benchmark_df['benchmark_fps'])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('FPS')
    plt.title('Hazır Model Detection FPS Karşılaştırması')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.bar(labels, benchmark_df['avg_person_count'])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Ortalama kişi/oyuncu sayısı')
    plt.title('Frame Başına Ortalama Detection Sayısı')
    plt.tight_layout()
    plt.show()
else:
    print('benchmark_df boş.')

## 11. Tracking video çıktısı üret

Burada amaç ID'lerin birkaç saniye korunup korunmadığını görmek. İlk olarak `ByteTrack`, sonra `BoT-SORT` denenir.

In [ ]:
TRACKING_OUT = WORK_ROOT / 'tracking_outputs'
TRACKING_OUT.mkdir(parents=True, exist_ok=True)

# İlk aşamada en iyi/uygun modeli kendin seçebilirsin.
# Otomatik olarak ilk yüklenen modeli kullanıyoruz.
SELECTED_MODEL_NAME = list(models.keys())[0] if models else None
SELECTED_MODEL = models[SELECTED_MODEL_NAME] if SELECTED_MODEL_NAME else None

print('Seçilen model:', SELECTED_MODEL_NAME)

if SELECTED_MODEL is not None and VIDEO_PATH.exists():
    for tracker in TRACKERS:
        run_name = f'{Path(SELECTED_MODEL_NAME).stem}_{tracker.replace(".yaml","")}_conf025'
        print('Track:', run_name)
        SELECTED_MODEL.track(
            source=str(VIDEO_PATH),
            conf=0.25,
            imgsz=640,
            classes=[PERSON_CLASS_ID],
            tracker=tracker,
            save=True,
            project=str(TRACKING_OUT),
            name=run_name,
            exist_ok=True
        )
else:
    print('Model veya video bulunamadı.')

## 12. Tracking analizi: ID sürekliliği

Bu hücre tracking sonucunu frame frame okuyup ID'lerin kaç frame boyunca korunduğunu ölçer.

Önemli kolonlar:

- `unique_track_ids`: video boyunca kaç farklı ID oluştu
- `avg_track_length_frames`: ID'lerin ortalama kaç frame sürdüğü
- `long_tracks_2_sec`: yaklaşık 2 saniyeden uzun süren ID sayısı
- `frames_with_ids_ratio`: frame'lerin ne kadarında ID üretilebildiği

In [ ]:
def analyze_tracking(model, video_path, model_name, tracker='bytetrack.yaml', conf=0.25, imgsz=640, max_frames=500):
    info = get_video_info(video_path)
    video_fps = info['fps'] if info['fps'] else 25
    track_lengths = Counter()
    frame_counts = []
    frames_with_ids = 0
    processed = 0

    results_iter = model.track(
        source=str(video_path),
        conf=conf,
        imgsz=imgsz,
        classes=[PERSON_CLASS_ID],
        tracker=tracker,
        stream=True,
        verbose=False,
        persist=True
    )

    for r in results_iter:
        if processed >= max_frames:
            break
        processed += 1
        ids = []
        if r.boxes is not None and r.boxes.id is not None:
            ids = [int(x) for x in r.boxes.id.cpu().numpy().tolist()]
            frames_with_ids += 1 if len(ids) > 0 else 0
            for tid in ids:
                track_lengths[tid] += 1
        frame_counts.append(len(ids))

    lengths = list(track_lengths.values())
    return {
        'model': model_name,
        'tracker': tracker,
        'conf': conf,
        'imgsz': imgsz,
        'processed_frames': processed,
        'unique_track_ids': len(track_lengths),
        'avg_ids_per_frame': float(np.mean(frame_counts)) if frame_counts else 0,
        'std_ids_per_frame': float(np.std(frame_counts)) if frame_counts else 0,
        'frames_with_ids_ratio': frames_with_ids / processed if processed else 0,
        'avg_track_length_frames': float(np.mean(lengths)) if lengths else 0,
        'median_track_length_frames': float(np.median(lengths)) if lengths else 0,
        'long_tracks_2_sec': int(sum(l >= 2 * video_fps for l in lengths)),
        'long_tracks_5_sec': int(sum(l >= 5 * video_fps for l in lengths))
    }

tracking_rows = []
if SELECTED_MODEL is not None and VIDEO_PATH.exists():
    for tracker in TRACKERS:
        row = analyze_tracking(
            SELECTED_MODEL, VIDEO_PATH, SELECTED_MODEL_NAME,
            tracker=tracker,
            conf=0.25,
            imgsz=640,
            max_frames=MAX_TRACK_ANALYSIS_FRAMES
        )
        tracking_rows.append(row)
        print(row)

tracking_df = pd.DataFrame(tracking_rows)
tracking_csv = WORK_ROOT / 'tracking_analysis_results.csv'
tracking_df.to_csv(tracking_csv, index=False)
display(tracking_df)
print('Kaydedildi:', tracking_csv)

## 13. Takım ayrımı için ilk hazır yapı: forma rengine göre clustering

Bu bölüm model eğitmez. Tracking/detection kutularından oyuncu crop'ları alır, HSV renk özellikleri çıkarır ve KMeans ile 2 takıma ayırmaya çalışır.

Bu yöntem baseline'dır. Hakem, kaleci, gölge, tribün ve yakın planlarda hata yapabilir.

In [ ]:
def crop_player_regions_from_video(model, video_path, max_crops=500, every_n_frames=15, conf=0.25, imgsz=640):
    cap = cv2.VideoCapture(str(video_path))
    crops = []
    meta = []
    frame_idx = 0
    while cap.isOpened() and len(crops) < max_crops:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % every_n_frames != 0:
            frame_idx += 1
            continue
        results = model.predict(
            source=frame,
            conf=conf,
            imgsz=imgsz,
            classes=[PERSON_CLASS_ID],
            verbose=False
        )
        r = results[0]
        if r.boxes is not None:
            boxes = r.boxes.xyxy.cpu().numpy().astype(int)
            confs = r.boxes.conf.cpu().numpy()
            for box, cf in zip(boxes, confs):
                x1, y1, x2, y2 = box
                h, w = frame.shape[:2]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w-1, x2), min(h-1, y2)
                if x2 <= x1 or y2 <= y1:
                    continue
                crop = frame[y1:y2, x1:x2]
                # Forma bölgesine odaklan: üst gövde yaklaşık orta-üst alan
                ch, cw = crop.shape[:2]
                torso = crop[int(ch*0.20):int(ch*0.65), int(cw*0.15):int(cw*0.85)]
                if torso.size == 0:
                    continue
                crops.append(torso)
                meta.append({'frame_idx': frame_idx, 'conf': float(cf), 'box': box.tolist()})
                if len(crops) >= max_crops:
                    break
        frame_idx += 1
    cap.release()
    return crops, meta


def hsv_feature(crop):
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    # Çok karanlık/puslu pikselleri azaltmak için S ve V filtresi
    mask = (hsv[:, :, 1] > 40) & (hsv[:, :, 2] > 40)
    pixels = hsv[mask]
    if len(pixels) < 20:
        pixels = hsv.reshape(-1, 3)
    # Ortalama H,S,V + H histogramı
    mean_hsv = pixels.mean(axis=0)
    hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten()
    hist_h = hist_h / (hist_h.sum() + 1e-6)
    return np.concatenate([mean_hsv, hist_h])

team_crops = []
team_features = []
team_meta = []

if SELECTED_MODEL is not None and VIDEO_PATH.exists():
    team_crops, team_meta = crop_player_regions_from_video(
        SELECTED_MODEL, VIDEO_PATH,
        max_crops=400,
        every_n_frames=15,
        conf=0.25,
        imgsz=640
    )
    print('Toplanan crop sayısı:', len(team_crops))
    if len(team_crops) >= 10:
        team_features = np.array([hsv_feature(c) for c in team_crops])
        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
        team_labels = kmeans.fit_predict(team_features)
        print('Takım cluster dağılımı:', Counter(team_labels))
    else:
        print('Takım ayrımı için yeterli crop yok.')

## 14. Takım crop örneklerini göster

In [ ]:
if len(team_crops) >= 10 and 'team_labels' in globals():
    for cluster_id in sorted(set(team_labels)):
        idxs = [i for i, lab in enumerate(team_labels) if lab == cluster_id][:12]
        plt.figure(figsize=(12, 4))
        for j, idx in enumerate(idxs):
            img = cv2.cvtColor(team_crops[idx], cv2.COLOR_BGR2RGB)
            plt.subplot(2, 6, j+1)
            plt.imshow(img)
            plt.axis('off')
        plt.suptitle(f'Cluster / Takım adayı: {cluster_id}')
        plt.tight_layout()
        plt.show()
else:
    print('Önce takım crop clustering hücresini çalıştır.')

## 15. Saha koordinatına dönüştürme için homography şablonu

Bu bölüm şimdilik şablondur. Gerçek saha analizi için video görüntüsündeki saha noktaları ile gerçek saha koordinatlarını eşleştirmek gerekir.

Örnek yaklaşım:

- Görüntü üzerinde 4 veya daha fazla saha noktası seçilir.
- Bu noktaların gerçek saha koordinatları girilir.
- `cv2.findHomography` ile dönüşüm matrisi hesaplanır.
- Oyuncu bbox'larının bottom-center noktası saha koordinatına çevrilir.

In [ ]:
# ÖRNEK / PLACEHOLDER
# img_points: video frame üzerindeki saha çizgi/köşe noktaları
# field_points: gerçek saha düzlemindeki karşılıkları

img_points = np.array([
    # [x_pixel, y_pixel],
    # [100, 700],
    # [1800, 700],
    # [1200, 300],
    # [700, 300],
], dtype=np.float32)

field_points = np.array([
    # [x_meter, y_meter],
    # [0, 68],
    # [105, 68],
    # [105, 0],
    # [0, 0],
], dtype=np.float32)

H = None
if len(img_points) >= 4 and len(field_points) >= 4:
    H, status = cv2.findHomography(img_points, field_points)
    print('Homography matrisi:')
    print(H)
else:
    print('Homography için en az 4 nokta girmen gerekiyor.')


def image_point_to_field(point_xy, H):
    p = np.array([[[point_xy[0], point_xy[1]]]], dtype=np.float32)
    transformed = cv2.perspectiveTransform(p, H)
    return transformed[0, 0]

## 16. Heatmap şablonu

Homography hazır olduğunda tracking noktalarını saha koordinatına çevirip heatmap çıkaracağız.

In [ ]:
def plot_heatmap(field_points_xy, field_w=105, field_h=68, bins=40):
    if len(field_points_xy) == 0:
        print('Heatmap için nokta yok.')
        return
    pts = np.array(field_points_xy)
    plt.figure(figsize=(12, 7))
    plt.hist2d(pts[:, 0], pts[:, 1], bins=bins, range=[[0, field_w], [0, field_h]])
    plt.colorbar(label='Yoğunluk')
    plt.xlim(0, field_w)
    plt.ylim(0, field_h)
    plt.xlabel('Saha X metre')
    plt.ylabel('Saha Y metre')
    plt.title('Oyuncu Konum Heatmap')
    plt.gca().invert_yaxis()
    plt.show()

# Örnek kullanım:
# plot_heatmap(field_positions)

## 17. Mesafe ve hız analizi şablonu

Tracking ID + homography ile her oyuncu için saha koordinatı elde edildiğinde mesafe ve hız hesaplanabilir.

In [ ]:
def compute_distance_speed(track_df, fps):
    # Beklenen kolonlar: track_id, frame_idx, field_x, field_y
    rows = []
    for tid, g in track_df.sort_values(['track_id', 'frame_idx']).groupby('track_id'):
        g = g.copy()
        dx = g['field_x'].diff()
        dy = g['field_y'].diff()
        dist = np.sqrt(dx**2 + dy**2).fillna(0)
        dt = g['frame_idx'].diff().fillna(1) / fps
        speed_mps = dist / dt.replace(0, np.nan)
        rows.append({
            'track_id': tid,
            'total_distance_m': float(dist.sum()),
            'avg_speed_mps': float(speed_mps.mean(skipna=True)),
            'max_speed_mps': float(speed_mps.max(skipna=True)),
            'observed_frames': len(g)
        })
    return pd.DataFrame(rows)

# Örnek kullanım:
# speed_df = compute_distance_speed(track_positions_df, fps=25)
# display(speed_df)

## 18. Sonuçları zip olarak indir

In [ ]:
ZIP_PATH = Path('/kaggle/working/ready_model_video_analysis_outputs.zip')
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

!zip -r /kaggle/working/ready_model_video_analysis_outputs.zip /kaggle/working/ready_model_video_analysis
print('Zip hazır:', ZIP_PATH)

# Kullanım sırası

1. `VIDEO_PATH` değerini düzelt.
2. Hazır modelleri yükle.
3. Örnek frame detection testi yap.
4. Video detection çıktılarını üret.
5. Benchmark tablosuna bak.
6. En uygun modeli seç.
7. ByteTrack ve BoT-SORT tracking testlerini yap.
8. ID sürekliliği tablosuna bak.
9. Takım ayrımı baseline hücrelerini çalıştır.
10. Daha sonra saha noktalarını girip homography + heatmap + mesafe/hız analizine geç.

İlk karar kriteri:

- Oyuncu kaçırıyorsa `conf=0.15` veya `imgsz=960` dene.
- Çok yanlış detection varsa `conf=0.35` dene.
- FPS çok düşükse `yolo11n.pt` veya `yolov8n.pt` + `imgsz=640` kullan.
- ID çok kopuyorsa ByteTrack ve BoT-SORT sonuçlarını karşılaştır.